In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_01/data'
os.listdir(data_dir)

['returns.csv', 'orders.csv', 'customers.csv', 'products.csv']

In [2]:
import pandas as pd
orders = pd.read_csv(os.path.join(data_dir, 'orders.csv'))
returns = pd.read_csv(os.path.join(data_dir, 'returns.csv'))
print("Orders columns:", orders.columns)
print("Returns columns:", returns.columns)
print(orders.head(2))
print(returns.head(2))

Orders columns: Index(['order_id', 'order_date', 'customer_id', 'product_id', 'quantity',
       'discount_pct', 'channel'],
      dtype='str')
Returns columns: Index(['return_id', 'order_id', 'return_date', 'returned_qty', 'reason'], dtype='str')
  order_id  order_date customer_id product_id  quantity  discount_pct  channel
0    O0001  2024-01-01         C01        P05         2            10    store
1    O0002  2024-01-01         C02        P08         3            15  partner
  return_id order_id return_date  returned_qty        reason
0      R001    O0009  2024-01-03             1  changed_mind
1      R002    O0018  2024-01-04             1       damaged


In [3]:
products = pd.read_csv(os.path.join(data_dir, 'products.csv'))
print("Products columns:", products.columns)
print(products.head(2))

Products columns: Index(['product_id', 'category', 'unit_price'], dtype='str')
  product_id category  unit_price
0        P01     Home        12.5
1        P02   Office        18.0


In [4]:
# Merge orders with products to get unit_price
orders_with_price = orders.merge(products, on='product_id', how='left')
orders_with_price['order_revenue'] = orders_with_price['quantity'] * orders_with_price['unit_price'] * (1 - orders_with_price['discount_pct'] / 100)

# Daily gross revenue
daily_gross = orders_with_price.groupby('order_date')['order_revenue'].sum().reset_index()
daily_gross.rename(columns={'order_date': 'date', 'order_revenue': 'gross_revenue'}, inplace=True)

# Merge returns with orders to get unit_price and discount_pct
returns_with_order = returns.merge(orders_with_price[['order_id', 'unit_price', 'discount_pct']], on='order_id', how='left')
returns_with_order['return_chargeback'] = returns_with_order['returned_qty'] * returns_with_order['unit_price'] * (1 - returns_with_order['discount_pct'] / 100)

# Daily return chargebacks
daily_returns = returns_with_order.groupby('return_date')['return_chargeback'].sum().reset_index()
daily_returns.rename(columns={'return_date': 'date'}, inplace=True)

# Merge daily gross and daily returns
daily_net = pd.merge(daily_gross, daily_returns, on='date', how='outer').fillna(0)
daily_net['net_revenue'] = daily_net['gross_revenue'] - daily_net['return_chargeback']

# Sort by date
daily_net = daily_net.sort_values('date').reset_index(drop=True)

# Calculate day-over-day increase
daily_net['dod_increase'] = daily_net['net_revenue'].diff()

print(daily_net)
max_increase_date = daily_net.loc[daily_net['dod_increase'].idxmax(), 'date']
print("Date with largest day-over-day increase:", max_increase_date)

          date  gross_revenue  return_chargeback  net_revenue  dod_increase
0   2024-01-01         640.90               0.00       640.90           NaN
1   2024-01-02          91.20               0.00        91.20       -549.70
2   2024-01-03         661.30              12.60       648.70        557.50
3   2024-01-04         147.50              12.60       134.90       -513.80
4   2024-01-05         610.15              34.20       575.95        441.05
5   2024-01-06          92.70              62.85        29.85       -546.10
6   2024-01-07         638.95              25.65       613.30        583.45
7   2024-01-08         137.05              85.40        51.65       -561.65
8   2024-01-09         640.90              57.20       583.70        532.05
9   2024-01-10          91.20              62.90        28.30       -555.40
10  2024-01-11         661.30              25.65       635.65        607.35
11  2024-01-12         147.50               0.00       147.50       -488.15
12  2024-01-

In [5]:
# Let's double check the logic.
# "Return chargebacks reduce revenue on the day the return is processed."
# "Compute daily net revenue and report the date with the largest day-over-day increase."

# Let's re-calculate daily net revenue.
# For each date, net revenue = (revenue from orders on that date) - (chargebacks from returns processed on that date)

# Let's check if there are any dates with returns but no orders, or vice versa.
all_dates = sorted(list(set(orders['order_date'].unique()).union(set(returns['return_date'].unique()))))
print("All dates:", all_dates)

# Create a dataframe with all dates
daily_net_rev = pd.DataFrame({'date': all_dates})

# Calculate gross revenue per day
gross_rev = orders_with_price.groupby('order_date')['order_revenue'].sum().reset_index()
gross_rev.columns = ['date', 'gross_revenue']

# Calculate return chargebacks per day
ret_chargebacks = returns_with_order.groupby('return_date')['return_chargeback'].sum().reset_index()
ret_chargebacks.columns = ['date', 'return_chargeback']

# Merge
daily_net_rev = daily_net_rev.merge(gross_rev, on='date', how='left').fillna(0)
daily_net_rev = daily_net_rev.merge(ret_chargebacks, on='date', how='left').fillna(0)

# Net revenue
daily_net_rev['net_revenue'] = daily_net_rev['gross_revenue'] - daily_net_rev['return_chargeback']

# Day-over-day increase
daily_net_rev['dod_increase'] = daily_net_rev['net_revenue'].diff()

print(daily_net_rev)
print("Max DoD increase date:", daily_net_rev.loc[daily_net_rev['dod_increase'].idxmax(), 'date'])

All dates: ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08', '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-12', '2024-01-13', '2024-01-14', '2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19', '2024-01-20', '2024-01-21', '2024-01-22', '2024-01-23', '2024-01-24', '2024-01-25', '2024-01-26', '2024-01-27', '2024-01-28', '2024-01-30']
          date  gross_revenue  return_chargeback  net_revenue  dod_increase
0   2024-01-01         640.90               0.00       640.90           NaN
1   2024-01-02          91.20               0.00        91.20       -549.70
2   2024-01-03         661.30              12.60       648.70        557.50
3   2024-01-04         147.50              12.60       134.90       -513.80
4   2024-01-05         610.15              34.20       575.95        441.05
5   2024-01-06          92.70              62.85        29.85       -546.10
6   2024-01-07         638.95              25.65  